## Extraction of images from videos

```
conda create -n ImgExtraction
conda activate ImgExtraction
pip install opencv-python
python3 -m ipykernel install --user --name ImgExtraction --display-name "Image extraction from videos"
```


## Anotación de imagenes

Las carpetas están organizadas de la siguiente manera

```
PUMA_CONCOLOR
  |
  +-- IMG001_imagen_001.jpg
  |
  +-- IMG001_imagen_002.jpg
```

In [ ]:
import os
import cv2
from datetime import datetime, timedelta
import ffmpeg  # Requires ffmpeg-python (install with `pip install ffmpeg-python`)
import piexif  # Requires piexif library (install with `pip install piexif`)
from concurrent.futures import ThreadPoolExecutor
from avi_r import AVIReader

def extract_images_from_video_with_exif(video_path, output_folder, video_file_name, target_duration=10, num_images=10):
    """
    Extract images from a video file, save them to the output folder, 
    and update the EXIF timestamp to match the video's frame time.
    """
    os.makedirs(output_folder, exist_ok=True)

    # Handle .AVI files
    if video_path.endswith(".AVI"):
        new_video_path = video_path[:-4] + ".avi"
        os.rename(video_path, new_video_path)
        video_path = new_video_path

    # Extract video creation timestamp using ffmpeg
    try:
        video_metadata = ffmpeg.probe(video_path)
        creation_time_str = next(
            stream['tags']['creation_time']
            for stream in video_metadata['streams']
            if 'tags' in stream and 'creation_time' in stream['tags']
        )
        # Convert to a datetime object
        video_creation_time = datetime.fromisoformat(creation_time_str.replace("Z", "+00:00"))
    except Exception as e:
        print(f"Warning: Could not extract creation time: {e}. Frames will be extracted without adding creation time metadata")
    
    # Determine if the video is an AVI file and use avi_r if necessary
    if video_path.endswith(".avi"):
        try:
            video = AVIReader(video_path)
            fps = video.frame_rate
            total_frames = int(video.num_frames)
        except Exception as e:
            print(f"Error reading AVI file with avi_r: {e}")
            return        
    else:
        video = cv2.VideoCapture(video_path)
        fps = video.get(cv2.CAP_PROP_FPS)
        total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Open video 
    video = cv2.VideoCapture(video_path)
    frames_to_capture = min(int(target_duration * fps), total_frames)
    interval = max(1, frames_to_capture // num_images)

    frame_count = 0
    captured_count = 0
    frame_duration = 1 / fps  # Duration of each frame in seconds

    while True:
        ret, frame = video.read()
        if not ret or captured_count >= num_images:
            break

        if frame_count % interval == 0:
            # Calculate timestamp for the current frame
            try:
                frame_timestamp = video_creation_time + timedelta(seconds=frame_count * frame_duration)
                exif_timestamp = frame_timestamp.strftime("%Y:%m:%d %H:%M:%S")  # EXIF-compliant format
            except:
                print(f'Warning: Error adding creation time, no video_creation_time detected')

            # Save the frame as an image
            image_filename = f"{video_file_name}_image{captured_count + 1:03d}.jpg"
            image_path = os.path.join(output_folder, image_filename)
            cv2.imwrite(image_path, frame)

            # Add EXIF timestamp metadata
            try:
                exif_dict = {"Exif": {piexif.ExifIFD.DateTimeOriginal: exif_timestamp.encode("utf-8")}}
                exif_bytes = piexif.dump(exif_dict)
                piexif.insert(exif_bytes, image_path)
                print(f"Captured: {image_path} with EXIF timestamp {exif_timestamp}")
            except Exception as e:
                print(f"Warning: Error adding EXIF data to {image_filename}: {e}")

            captured_count += 1

        frame_count += 1

    video.release()
    print(f"Extraction complete. {captured_count} images saved to {output_folder}.")

def process_directory(input_dir, output_root):
    """
    Process all videos in a given directory, saving extracted frames to a new folder.
    Handles duplicate filenames with different extensions.
    """
    video_extensions = {".mp4", ".avi", ".mov", ".mkv"}  # Add more extensions as needed
    species_name = os.path.basename(input_dir)
    output_folder = os.path.join(output_root, f"{species_name}_extracted")
    os.makedirs(output_folder, exist_ok=True)  # Create output folder if it doesn't exist

    video_filenames = set()  # Set to track unique filenames

    for file_name in os.listdir(input_dir):
        file_path = os.path.join(input_dir, file_name)
        if os.path.isfile(file_path) and os.path.splitext(file_name)[1].lower() in video_extensions:
            base_name, ext = os.path.splitext(file_name)
            if base_name in video_filenames:
                # Handle duplicate filenames with different extensions
                video_file_name = f"{base_name}_{ext[1:]}" 
            else:
                video_file_name = base_name
            video_filenames.add(base_name) 
            extract_images_from_video_with_exif(file_path, output_folder, video_file_name)

def main(input_root, output_root, max_workers=4):
    """
    Main function to process all subdirectories in parallel.
    """
    os.makedirs(output_root, exist_ok=True)
    subdirs = [os.path.join(input_root, d) for d in os.listdir(input_root) if os.path.isdir(os.path.join(input_root, d))]

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(process_directory, subdir, output_root) for subdir in subdirs]
        for future in futures:
            future.result()  # Wait for all tasks to complete

if __name__ == "__main__":
   input_root = "/mnt/STORAGE/csar/pipo_images"  # Replace with the path to your main directory
   output_root = "../data/images_from_videos"  # Replace with the path to save extracted frames
   main(input_root, output_root, max_workers=20)

# if __name__ == "__main__":
#    input_root = "/mnt/STORAGE/sofia/test_folder"  # Replace with the path to your main directory
#    output_root = "../data/test_avi"  # Replace with the path to save extracted frames
#    main(input_root, output_root, max_workers=20)

Captured: ../data/images_from_videos/PUMA_CONCOLOR_2022_extracted/IMG_0015_image001.jpg with EXIF timestamp 2022:07:20 09:02:57
Captured: ../data/images_from_videos/PUMA_CONCOLOR_2022_extracted/IMG_0015_image002.jpg with EXIF timestamp 2022:07:20 09:02:58
Captured: ../data/images_from_videos/ROEDORES_extracted/IMG_0019_image001.jpg with EXIF timestamp 2022:09:30 05:35:08
Captured: ../data/images_from_videos/PUMA_CONCOLOR_2022_extracted/IMG_0015_image003.jpg with EXIF timestamp 2022:07:20 09:02:59
Captured: ../data/images_from_videos/PECARI_TAJACU_2022_extracted/IMG_0015_image001.jpg with EXIF timestamp 2022:07:04 22:05:16
Captured: ../data/images_from_videos/PUMA_CONCOLOR_2022_extracted/IMG_0015_image004.jpg with EXIF timestamp 2022:07:20 09:03:00
Captured: ../data/images_from_videos/SYLVILAGUS_SP_extracted/IMG_0029 (2)_image001.jpg with EXIF timestamp 2022:05:23 09:06:28
Captured: ../data/images_from_videos/LYNX_RUFUS_extracted/IMG_0049 (2)_image001.jpg with EXIF timestamp 2022:06:19 

## Revisar que todos los archivos hayan sido extraidos

Solo hubo problemas en los archivos de las aves. No entiendo bien por que. 

In [52]:
import os
import pandas as pd
import shutil


In [71]:
# Revisar que todos los videos hayan sido extraidos

root_dir = '/mnt/STORAGE/csar/pipo_images'

# Initialize a list to store the data
data = []

# Walk through the filesystem
for parent, species_dirs, _ in os.walk(root_dir):
    for species in species_dirs:
        species_path = os.path.join(parent, species)
        #print(species_path)
        for image in os.listdir(species_path):
            image_path = os.path.join(species_path, image)
            if os.path.isfile(image_path):  # Check if it's a file (image)
                # Append the four levels to the list
                data.append([os.path.basename(root_dir),  # Parent folder name
                                    species,                    # Species folder name
                                    image])                     # Image file name

# Create a DataFrame from the collected data
original_df = pd.DataFrame(data, columns=["Parent_Folder", "Species", "Image"])
original_df_videos = original_df[~original_df['Image'].str.contains('.JPG')]
original_df_videos.loc[:, 'Original_file'] = original_df['Image'].str.replace(r'\.\w*$', '', regex=True)

original_df_videos_filtered = original_df_videos[~original_df_videos['Image'].str.contains('.M4V')]

print(original_df['Image'].str.split('.', expand = True)[1].unique())
print(original_df_videos['Image'].str.split('.', expand = True)[1].unique())
print(original_df_videos_filtered[original_df_videos_filtered['Species'] == 'AVES_2022'].head())
print(original_df_videos_filtered.shape)

['MP4' 'avi' 'JPG' 'M4V' 'AVI']
['MP4' 'avi' 'M4V' 'AVI']
   Parent_Folder    Species         Image Original_file
12   pipo_images  AVES_2022  IMG_0184.avi      IMG_0184
13   pipo_images  AVES_2022  IMG_0106.avi      IMG_0106
16   pipo_images  AVES_2022  IMG_0297.MP4      IMG_0297
17   pipo_images  AVES_2022  IMG_0478.MP4      IMG_0478
18   pipo_images  AVES_2022  IMG_0471.MP4      IMG_0471
(1802, 4)


/tmp/ipykernel_21012/2601685913.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  original_df_videos.loc[:, 'Original_file'] = original_df['Image'].str.replace(r'\.\w*$', '', regex=True)


In [82]:
# Define the root directory (change this to your actual root folder path)
root_dir = "../data/images_from_videos"

# Initialize a list to store the data
data = []

# Walk through the filesystem
for parent, species_dirs, _ in os.walk(root_dir):
    for species in species_dirs:
        species_path = os.path.join(parent, species)
        #print(species_path)
        for image in os.listdir(species_path):
            image_path = os.path.join(species_path, image)
            if os.path.isfile(image_path):  # Check if it's a file (image)
                # Append the four levels to the list
                data.append([os.path.basename(root_dir),  # Parent folder name
                                    species,                    # Species folder name
                                    image])                     # Image file name

# Create a DataFrame from the collected data
extracted_df = pd.DataFrame(data, columns=["Parent_Folder", "Species", "Image"])
extracted_df[['Original_file','img']] = extracted_df['Image'].str.split(r'_(?=[Ii]mage)', n=1, expand=True)
extracted_df['Sp'] = extracted_df['Species'].str.replace('_extracted','')

extracted_df.shape


(15290, 6)

In [73]:
# Revisar que haya el mismo numero de archivos con nombre original antes y despues de la extracción.

counts_extracted = extracted_df.groupby(['Species','Original_file']).count().reset_index()
counts_extracted['Original_sp'] = counts_extracted['Species'].str.replace('_extracted','')

print(counts_extracted.groupby(['Species']).count()['Original_file'])

counts_original = original_df_videos_filtered[['Species','Image']].groupby(['Species']).count()

#print(counts_extracted)
print(counts_original)

Species
AVES_2022_extracted                      291
BASSSEISCUS_ASTUTUS_extracted              5
CONEPATUS_LEUCONOTUS_2022_extracted       39
DIDELPHIS_VIRGINIATA_2022_extracted       28
GANADO_2022_extracted                    312
LYNX_RUFUS_extracted                      16
MEPITIS_MACROURA_2022_extracted           80
NASUA_NARICA_2022_extracted                4
ODOCOILEUS_VIRGINIANUS_2022_extracted     45
OTOSPERMOPHILUS_VERIEGATUS_extracted       4
PECARI_TAJACU_2022_extracted             108
PERROS_extracted                           2
PUMA_CONCOLOR_2022_extracted              35
REPTILES_extracted                         3
ROEDORES_extracted                        12
SCIURUS_OCOLATUS_extracted                43
SPILOGALE_GRACILIS_2022_extracted          5
SYLVILAGUS_SP_extracted                  206
UROCYON_CINEREORGENTEUS_extracted        291
Name: Original_file, dtype: int64
                             Image
Species                           
AVES_2022                      56

In [83]:
extracted_df['Path'] = extracted_df.apply(lambda row: os.path.join('../data/',row['Parent_Folder'], row['Species'], row['Image']), axis=1)
extracted_df.head()

,Parent_Folder,Species,Image,Original_file,img,Sp,Path
0,images_from_videos,DIDELPHIS_VIRGINIATA_2022_extracted,IMG_0009_image010.jpg,IMG_0009,image010.jpg,DIDELPHIS_VIRGINIATA_2022,../data/images_from_videos/DIDELPHIS_VIRGINIAT...
1,images_from_videos,DIDELPHIS_VIRGINIATA_2022_extracted,IMG_0063_image008.jpg,IMG_0063,image008.jpg,DIDELPHIS_VIRGINIATA_2022,../data/images_from_videos/DIDELPHIS_VIRGINIAT...
2,images_from_videos,DIDELPHIS_VIRGINIATA_2022_extracted,IMG_0252_image003.jpg,IMG_0252,image003.jpg,DIDELPHIS_VIRGINIATA_2022,../data/images_from_videos/DIDELPHIS_VIRGINIAT...
3,images_from_videos,DIDELPHIS_VIRGINIATA_2022_extracted,IMG_0094_image002.jpg,IMG_0094,image002.jpg,DIDELPHIS_VIRGINIATA_2022,../data/images_from_videos/DIDELPHIS_VIRGINIAT...
4,images_from_videos,DIDELPHIS_VIRGINIATA_2022_extracted,IMG_0063_image007.jpg,IMG_0063,image007.jpg,DIDELPHIS_VIRGINIATA_2022,../data/images_from_videos/DIDELPHIS_VIRGINIAT...


In [84]:
extracted_df.to_csv('../data/images_from_videos_metadata.csv', index=False)

# Detección de imagenes

Muchas de las imagenes que se extrajeron no tienen animales por lo que es necesario filtrarlas. 

**NOTA:** Hay que cambiar de Kernel! USAR pytorch_wildlife

In [1]:
import os
import pandas as pd
import torch
from PytorchWildlife.models import detection as pw_detection
from PytorchWildlife import utils as pw_utils

DEVICE = 'cpu'

# Initializing the MegaDetectorV6 model for image detection
detection_model = pw_detection.MegaDetectorV6(device=DEVICE, pretrained=True, version="yolov9c")

# Function to extract detections from each image. The images can be joined with species metadata by path and image name
def extract_data_from_detections(results):
    """
    Extracts data from a list of detection results and creates a DataFrame.

    Parameters:
    results (list of dict): A list of dictionaries where each dictionary contains:
        - 'img_id' (str): The image ID.
        - 'labels' (list of str): A list of labels, where each label is a string containing two words: 
          the label name and the certainty score separated by a space.

    Returns:
    pd.DataFrame: A DataFrame with the following columns:
        - 'img_id' (str): The image ID.
        - 'label' (str): The label name.
        - 'certainty' (float): The certainty score.
    """
    
    # List to store the extracted data
    extracted_data = []

    # Extract the keys 'labels' and 'img_id'
    for d in results:
        img_id = d['img_id']
        for label in d['labels']:
            label_name, certainty = label.split()
            extracted_data.append({'img_id': img_id, 'label': label_name, 'certainty': float(certainty)})

    # Create DataFrame
    df = pd.DataFrame(extracted_data)
    return df


Ultralytics 8.3.48 🚀 Python-3.8.20 torch-2.4.1+cu121 CPU (Intel Xeon E5-2697 v4 2.30GHz)
YOLOv9c summary (fused): 384 layers, 25,321,561 parameters, 0 gradients, 102.3 GFLOPs


In [4]:
# Detect for all folders in mod_images_from_videos

results_per_species = []
all_images_detection_metadata = pd.DataFrame()

# Define the path where results will be saved
out_path = '../data/img_from_video_metadata/'

# Ensure the folder exists
os.makedirs(out_path, exist_ok=True)

for species in os.listdir('../data/images_from_videos'):

    print(f'Detecting images from {species}')
    
    folder_path = os.path.join("..","data","images_from_videos",species)
    results = detection_model.batch_image_detection(folder_path, batch_size=16)
    results_per_species.append(results)

    metadata = extract_data_from_detections(results)
    metadata.to_csv(os.path.join(out_path, species + '.csv'))


Detecting images from DIDELPHIS_VIRGINIATA_2022_extracted


  0%|                                                                                                                                                  | 0/18 [00:00<?, ?it/s]


0: 640x640 1 animal, 413.0ms
1: 640x640 1 animal, 413.0ms
2: 640x640 1 animal, 413.0ms
3: 640x640 1 animal, 413.0ms
4: 640x640 1 animal, 413.0ms
5: 640x640 1 animal, 413.0ms
6: 640x640 1 animal, 413.0ms
7: 640x640 1 animal, 413.0ms
8: 640x640 1 animal, 413.0ms
9: 640x640 1 animal, 413.0ms
10: 640x640 1 animal, 413.0ms
11: 640x640 1 animal, 413.0ms
12: 640x640 1 animal, 413.0ms
13: 640x640 1 animal, 413.0ms
14: 640x640 (no detections), 413.0ms
15: 640x640 (no detections), 413.0ms
Speed: 3.1ms preprocess, 413.0ms inference, 0.4ms postprocess per image at shape (16, 3, 640, 640)


  6%|███████▋                                                                                                                                  | 1/18 [00:07<02:03,  7.26s/it]


0: 640x640 (no detections), 408.5ms
1: 640x640 (no detections), 408.5ms
2: 640x640 (no detections), 408.5ms
3: 640x640 (no detections), 408.5ms
4: 640x640 1 animal, 408.5ms
5: 640x640 1 animal, 408.5ms
6: 640x640 1 animal, 408.5ms
7: 640x640 1 animal, 408.5ms
8: 640x640 1 animal, 408.5ms
9: 640x640 1 animal, 408.5ms
10: 640x640 (no detections), 408.5ms
11: 640x640 (no detections), 408.5ms
12: 640x640 (no detections), 408.5ms
13: 640x640 1 animal, 408.5ms
14: 640x640 2 animals, 408.5ms
15: 640x640 1 animal, 408.5ms
Speed: 2.5ms preprocess, 408.5ms inference, 0.3ms postprocess per image at shape (16, 3, 640, 640)


...

 80%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████▍                           | 8/10 [00:43<00:12,  6.23s/it]


0: 640x640 1 animal, 425.8ms
1: 640x640 1 animal, 425.8ms
2: 640x640 1 animal, 425.8ms
3: 640x640 1 animal, 425.8ms
4: 640x640 1 animal, 425.8ms
5: 640x640 1 animal, 425.8ms
6: 640x640 1 animal, 425.8ms
7: 640x640 2 animals, 425.8ms
8: 640x640 1 animal, 425.8ms
9: 640x640 1 animal, 425.8ms
10: 640x640 (no detections), 425.8ms
11: 640x640 (no detections), 425.8ms
12: 640x640 1 animal, 425.8ms
13: 640x640 3 animals, 425.8ms
14: 640x640 2 animals, 425.8ms
15: 640x640 1 animal, 425.8ms
Speed: 2.9ms preprocess, 425.8ms inference, 0.4ms postprocess per image at shape (16, 3, 640, 640)


 90%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▏             | 9/10 [00:50<00:06,  6.61s/it]


0: 640x640 2 animals, 424.3ms
1: 640x640 3 animals, 424.3ms
2: 640x640 4 animals, 424.3ms
3: 640x640 2 animals, 424.3ms
4: 640x640 6 animals, 424.3ms
5: 640x640 2 animals, 424.3ms
6: 640x640 1 animal, 424.3ms
7: 640x640 1 animal, 424.3ms
8: 640x640 1 animal, 424.3ms
9: 640x640 1 animal, 424.3ms
10: 640x640 1 animal, 424.3ms
11: 640x640 1 animal, 424.3ms
12: 640x640 1 animal, 424.3ms
13: 640x640 1 animal, 424.3ms
14: 640x640 1 animal, 424.3ms
15: 640x640 1 animal, 424.3ms
Speed: 2.5ms preprocess, 424.3ms inference, 0.5ms postprocess per image at shape (16, 3, 640, 640)



00%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:58<00:00,  5.81s/it]

## Seleccion de imagenes con animales

Vamos a seleccionar las imagenes en las que si se detectó un animal y generar la tabla de metadatos para cada especie

In [5]:
import pandas as pd
import os 

In [12]:
file_list = os.listdir('../data/img_from_video_metadata')

dfs = [pd.read_csv(os.path.join('../data/img_from_video_metadata',file)) for file in file_list]

In [15]:
df = pd.concat(dfs)
df

,Unnamed: 0,img_id,label,certainty
0,0,../data/mod_images_from_videos/CONEPATUS_LEUCO...,animal,0.86
1,1,../data/mod_images_from_videos/CONEPATUS_LEUCO...,animal,0.68
2,2,../data/mod_images_from_videos/CONEPATUS_LEUCO...,animal,0.47
3,3,../data/mod_images_from_videos/CONEPATUS_LEUCO...,animal,0.36
4,4,../data/mod_images_from_videos/CONEPATUS_LEUCO...,animal,0.48
...,...,...,...,...
1048,1048,../data/mod_images_from_videos/ODOCOILEUS_VIRG...,animal,0.93
1049,1049,../data/mod_images_from_videos/ODOCOILEUS_VIRG...,animal,0.93
1050,1050,../data/mod_images_from_videos/ODOCOILEUS_VIRG...,animal,0.92
1051,1051,../data/mod_images_from_videos/ODOCOILEUS_VIRG...,animal,0.92
